# Large-Scale Data Management -
# Assignment 2

**NEGI HOXHA**  
**April 2026**

## 1 Introduction

This report presents the implementation of a large-scale data management pipeline using Apache Kafka, Apache Spark Structured Streaming, and Apache Cassandra.

The goal of the project is to generate a stream of songs listened to by a group of people, process this stream with Spark, enrich it using song metadata from `spotify-songs.csv`, and persist the final records into Cassandra.

The implementation is divided into two main parts:

- **Part 1:** A Python script that simulates song listening events and sends them to a Kafka topic.
- **Part 2:** A PySpark script that consumes the stream, processes and enriches the records, and writes them to Cassandra.

## 2 Part 1: Kafka Producer

The first part involves a Python script that simulates streaming song listening data by generating random song events for a group of users and sending them to a Kafka topic named `spotify_songs`.

The script uses the Faker library to generate at least 10 random names and also includes my own name, `NEGI HOXHA`. Song titles are read from `spotify-songs.csv`. Each Kafka message contains:

- `person_name`
- `song_name`
- `listened_at`

### 2.1 Python Code

In [ ]:
import csv
import json
import asyncio
import random
from datetime import datetime, timezone

from aiokafka import AIOKafkaProducer
from faker import Faker

TOPIC = "spotify_songs"
BOOTSTRAP_SERVERS = "localhost:29092"
MY_NAME = "NEGI HOXHA"
MIN_DELAY_SECONDS = 5
MAX_DELAY_SECONDS = 15

fake = Faker()


def serializer(value):
    return json.dumps(value).encode("utf-8")


def load_song_names(csv_path="spotify-songs.csv"):
    songs = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            song_name = row["name"].strip()
            if song_name:
                songs.append(song_name)
    return songs


async def produce():
    producer = AIOKafkaProducer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        value_serializer=serializer,
        compression_type="gzip"
    )

    songs = load_song_names()

    people = [fake.name() for _ in range(10)]
    people.append(MY_NAME)

    await producer.start()
    try:
        while True:
            for person in people:
                song = random.choice(songs)
                data = {
                    "person_name": person,
                    "song_name": song,
                    "listened_at": datetime.now(timezone.utc).isoformat()
                }
                await producer.send_and_wait(TOPIC, data)
                print(data)
            await asyncio.sleep(random.randint(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))
    finally:
        await producer.stop()


if __name__ == "__main__":
    asyncio.run(produce())

The producer sends one event per user in each cycle and then waits a few seconds before generating the next batch of events.

## 3 Part 2: Spark Streaming Consumer

The second part involves a PySpark Structured Streaming script that consumes the Kafka messages from topic `spotify_songs`, parses the JSON records, derives the listening hour, joins the stream with the static `spotify-songs.csv` dataset, and writes the enriched rows to Cassandra.

The persistence interval is configurable through a variable, and in this implementation it is set to 30 seconds, as requested in the assignment.

### 3.1 PySpark Code

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import (
    from_json, col, to_timestamp, date_format
)

TOPIC = "spotify_songs"
BOOTSTRAP_SERVERS = "localhost:29092"
CSV_PATH = "/vagrant/spotify-songs.csv"
CHECKPOINT_DIR = "/tmp/spotify_checkpoint"
TRIGGER_SECONDS = 30

schema = StructType([
    StructField("person_name", StringType(), False),
    StructField("song_name", StringType(), False),
    StructField("listened_at", StringType(), False),
])

spark = (
    SparkSession.builder
    .appName("SpotifySongStreamingToCassandra")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
            "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"
        ])
    )
    .config("spark.cassandra.connection.host", "localhost")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

songs_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
    .select(
        col("name").alias("csv_song_name"),
        col("artists"),
        col("duration_ms"),
        col("album_name"),
        col("album_release_date"),
        col("danceability"),
        col("energy"),
        col("key"),
        col("loudness"),
        col("mode"),
        col("speechiness"),
        col("acousticness"),
        col("instrumentalness"),
        col("liveness"),
        col("valence"),
        col("tempo")
    )
    .dropDuplicates(["csv_song_name"])
)

songs_df.cache()
songs_df.count()

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

parsed_df = (
    kafka_df
    .selectExpr("CAST(value AS STRING) AS json_value")
    .select(from_json(col("json_value"), schema).alias("data"))
    .select("data.*")
    .withColumn("listened_at_ts", to_timestamp(col("listened_at")))
    .withColumn("listen_hour", date_format(col("listened_at_ts"), "yyyy-MM-dd HH:00"))
)

enriched_df = (
    parsed_df.join(
        songs_df,
        parsed_df.song_name == songs_df.csv_song_name,
        "inner"
    )
    .select(
        col("person_name"),
        col("listen_hour"),
        col("listened_at_ts").alias("listened_at"),
        col("song_name"),
        col("artists"),
        col("duration_ms"),
        col("album_name"),
        col("album_release_date"),
        col("danceability"),
        col("energy"),
        col("key"),
        col("loudness"),
        col("mode"),
        col("speechiness"),
        col("acousticness"),
        col("instrumentalness"),
        col("liveness"),
        col("valence"),
        col("tempo")
    )
)

def write_to_cassandra(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    (
        batch_df.write
        .format("org.apache.spark.sql.cassandra")
        .mode("append")
        .options(table="person_hour_song_history", keyspace="spotify")
        .save()
    )

query = (
    enriched_df.writeStream
    .foreachBatch(write_to_cassandra)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime=f"{TRIGGER_SECONDS} seconds")
    .start()
)

query.awaitTermination()

The static songs dataset is cached before stream processing in order to improve performance, since it is reused in every micro-batch.

## 4 Cassandra Data Model

To optimize queries for a particular person and hour, I created the Cassandra table `person_hour_song_history` in keyspace `spotify`.

The table schema is designed so that all songs listened to by one person during one hour are stored in the same partition. This makes it efficient to query:

- the average danceability for a person during a specific hour
- the names of songs listened to by a person during a specific hour

```sql
CREATE KEYSPACE IF NOT EXISTS spotify
WITH replication = {'class':'SimpleStrategy', 'replication_factor':1};

CREATE TABLE person_hour_song_history (
    person_name text,
    listen_hour text,
    listened_at timestamp,
    song_name text,
    artists text,
    duration_ms int,
    album_name text,
    album_release_date text,
    danceability double,
    energy double,
    key int,
    loudness double,
    mode int,
    speechiness double,
    acousticness double,
    instrumentalness double,
    liveness double,
    valence double,
    tempo double,
    PRIMARY KEY ((person_name, listen_hour), listened_at, song_name)
) WITH CLUSTERING ORDER BY (listened_at ASC, song_name ASC);
```

The partition key is `(person_name, listen_hour)`, while `listened_at` and `song_name` are clustering columns. This design supports the exact query pattern required by the assignment, namely aggregations and lookups for a specific person during a specific hour.

## 5 Sample Persisted Rows

Below is a sample of persisted rows from the Cassandra table.

```sql
SELECT person_name, listen_hour, listened_at, song_name, artists, danceability, energy, tempo
FROM spotify.person_hour_song_history
LIMIT 50;


 person_name | listen_hour      | listened_at                     | song_name                         | artists                                     | danceability | energy | tempo
-------------+------------------+---------------------------------+-----------------------------------+---------------------------------------------+--------------+--------+---------
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:11:42.120000+0000 |                            CHOUHA |                              Bo9al, ASHE 22 |        0.906 |  0.446 | 139.966
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:11:57.175000+0000 |             Lonely This Christmas |                                         Mud |        0.551 |  0.325 |  98.932
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:12:05.224000+0000 |       MONEY ON THE DASH - SPED UP |                         Elley Duhé, Whethan |        0.777 |  0.816 |  150.09
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:12:14.303000+0000 |                          Žiburiai |                                 Jessica Shy |        0.663 |  0.422 |   95.05
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:12:29.356000+0000 |                      Miss my Home |                                   Nimo, Eno |        0.726 |   0.73 |  94.038
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:12:39.403000+0000 |                            ВМЕСТЕ |                          Aarne, BUSHIDO ZHO |        0.772 |  0.785 |  143.91
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:12:46.449000+0000 |                             BELLA |                                     SIDARTA |        0.757 |  0.563 |  84.017
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:01.520000+0000 |                            Stúfur |                     Baggalútur, Friðrik Dór |        0.671 |  0.667 | 120.859
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:11.605000+0000 |                            Lavida |                                       Pryme |        0.734 |   0.64 | 111.732
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:21.649000+0000 |                         Stockholm |                                Peg Parnevik |        0.436 |  0.388 | 151.671
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:29.700000+0000 |                             Intro |                                     Luciano |        0.642 |   0.58 | 140.161
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:34.745000+0000 |                      Nada Con Vos |                        Paloma Lomax, Sharik |        0.879 |  0.544 |  91.988
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:45.795000+0000 |                         Łakomstwo |                                     PRO8L3M |        0.393 |  0.686 | 161.426
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:52.846000+0000 |              Afară Ninge Liniştit |                               Stefan Hrusca |         0.45 |  0.135 |  97.295
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:13:58.898000+0000 |                          Culiacan |      Engel Montaz, Kuv507, Latinnites Music |        0.701 |  0.567 |  144.99
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:14:10.975000+0000 |                        Terasa Ada |                              Sufian Suhaimi |        0.489 |  0.644 | 180.015
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:14:24.022000+0000 |                              อิจฉา |                                       MEYOU |         0.55 |   0.54 | 136.167
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:14:39.073000+0000 |                          Ape Song |                                Viktor Sheen |        0.684 |  0.535 | 179.904
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:14:46.116000+0000 |                         Mr Reggae |                                      L.A.B. |        0.829 |  0.701 | 164.035
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:00.175000+0000 |               The Real Slim Shady |                                      Eminem |        0.949 |  0.661 | 104.504
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:05.219000+0000 |                       Carry me go |                                       Jvson |        0.872 |  0.487 | 113.045
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:17.273000+0000 |           Un Millón de Primaveras |                           Vicente Fernández |         0.73 |  0.341 | 100.558
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:28.324000+0000 |                  ياحلوه يا مظلومه |                                    ابو سراج |        0.544 |  0.992 | 149.957
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:35.368000+0000 |       Hit Me Up (feat. Nomovodka) |                             Binz, Nomovodka |        0.775 |   0.44 |  99.993
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:47.415000+0000 |                       Hautajaiset |                          Gettomasa, Sexmane |        0.666 |  0.705 | 174.145
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:15:58.455000+0000 |             Anestesiado - Ao Vivo |                                 Murilo Huff |         0.53 |  0.855 | 159.966
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:16:13.488000+0000 |                   Downers At Dusk |                          Talha Anjum, Umair |        0.856 |  0.447 | 120.069
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:16:24.533000+0000 |                        Last of us |                           MiyaGi & Endspiel |         0.67 |   0.53 |   76.97
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:16:36.581000+0000 |                     Waste No Time |                        Siles Hendrix, Ekene |        0.914 |  0.526 |  93.005
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:16:41.624000+0000 |                        Baarishein |                                   Anuv Jain |        0.476 |  0.124 |  94.299
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:16:50.667000+0000 |             Chooky - Busta Rhymes |            O.T, Busta Rhymes, Elesia Iimura |        0.851 |  0.782 |  95.983
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:17:05.725000+0000 |                            Я и ты |                                       JAPYR |        0.859 |  0.289 |  129.91
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:17:18.777000+0000 |                        Na Contigo |                           Chimbala, Darmiko |        0.908 |  0.736 | 127.981
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:17:28.821000+0000 |                   Zihaal e Miskin | Javed-Mohsin, Vishal Mishra, Shreya Ghoshal |        0.581 |  0.632 | 128.844
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:17:42.874000+0000 |                        Пару минут |                                     Jaman T |        0.742 |  0.398 | 130.024
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:17:56.924000+0000 |                           Mi Amor |                        Sharn, 40k, The Paul |        0.637 |  0.703 |   81.99
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:18:05.963000+0000 |                        ):阿修羅:( |                                    King Gnu |         0.62 |  0.856 | 170.004
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:18:12.006000+0000 |                    Piano Merengue |                                     Damiron |        0.684 |  0.465 | 123.257
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:18:22.049000+0000 |                      Florin Salam |                    Tzanca Uraganu, Mr. Juve |        0.661 |  0.846 |  175.99
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:18:33.084000+0000 | Restiamo Soli (feat. Fabri Fibra) |                       Gemitaiz, Fabri Fibra |        0.637 |  0.733 | 188.183
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:18:41.124000+0000 |                        Konstnärer |                                 Asme, Shiro |        0.728 |  0.597 | 105.121
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:18:55.167000+0000 |          Ballon d'Or (feat. Nayt) |                              Gemitaiz, Nayt |        0.801 |  0.682 | 104.958
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:19:07.212000+0000 |                       JET_Set.mp3 |                        Emilia, NATHY PELUSO |        0.658 |  0.798 | 135.515
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:19:15.255000+0000 |                         Me Escapé |                DobleP, Lauty Gram, Gusty dj |        0.844 |   0.68 | 100.109
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:19:24.305000+0000 |               Faz Completo, Chefe |                                Mc IG, DJ WN |        0.714 |  0.373 | 165.935
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:19:38.348000+0000 |                             Jinja |                                     Olamide |        0.898 |  0.735 |  95.992
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:19:47.389000+0000 |                   I know who i am |                                       Jvson |        0.743 |  0.596 | 114.002
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:19:54.438000+0000 |                     Разбити мечти |                               Lidia, Debora |        0.592 |  0.942 | 167.947
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:20:00.481000+0000 |       Astronaut - Spotify Singles |                                     Bolaget |        0.632 |   0.82 | 125.937
  Jay Walker | 2026-04-02 16:00 | 2026-04-02 16:20:05.539000+0000 |                           Navidad |                            Los Toribianitos |        0.668 |   0.56 | 143.159
```

## 6 Querying Cassandra

The following Cassandra queries were executed for my own name and a particular hour.

### Query 1: Average danceability for my name and a particular hour

```sql
cqlsh:spotify> SELECT AVG(danceability)
           ... FROM spotify.person_hour_song_history
           ... WHERE person_name = 'NEGI HOXHA'
           ...   AND listen_hour = '2026-04-02 16:00';

 system.avg(danceability)
--------------------------
                 0.655569

(1 rows)
cqlsh:spotify> SELECT listened_at, song_name
           ... FROM spotify.person_hour_song_history
           ... WHERE person_name = 'NEGI HOXHA'
           ...   AND listen_hour = '2026-04-02 16:00';

 listened_at                     | song_name
---------------------------------+-----------------------------------------------------------------------------
 2026-04-02 16:11:42.146000+0000 |                                                              đưa em về nhàa
 2026-04-02 16:11:57.201000+0000 |                                                                  Be the One
 2026-04-02 16:12:14.325000+0000 |                                                        Trái đất ôm Mặt trời
 2026-04-02 16:12:29.381000+0000 |                                                                      MOTORA
 2026-04-02 16:12:39.435000+0000 |                   love nwantiti (feat. ElGrande Toto) - North African Remix
 2026-04-02 16:12:46.478000+0000 |                                                        Sinterklaas Is Jarig
 2026-04-02 16:13:01.581000+0000 |                                                                     Bertaut
 2026-04-02 16:13:11.628000+0000 |                                                                  You and Me
 2026-04-02 16:13:21.679000+0000 |                                                                 Santa Lucia
 2026-04-02 16:13:29.724000+0000 |                                                                       kunim
 2026-04-02 16:13:34.773000+0000 |                                 MY LOVE SONG 2 (feat. Coez & Frah Quintale)
 2026-04-02 16:13:45.822000+0000 |                                                        Těšíme Se Na Ježíška
 2026-04-02 16:13:52.877000+0000 |                                                                  Be the One
 2026-04-02 16:13:58.949000+0000 |                                                         La route est longue
 2026-04-02 16:14:11.000000+0000 |                                                                  NAPOLITANO
 2026-04-02 16:14:24.045000+0000 |                                                        你不會一輩子的愛上我
 2026-04-02 16:14:39.097000+0000 |                                                          Un Loquito Como Yo
 2026-04-02 16:14:46.141000+0000 | Popular (with Playboi Carti & Madonna) - Music from the HBO Original Series
 2026-04-02 16:15:00.200000+0000 |                                       Forgive Our Trespasses (feat. Demola)
 2026-04-02 16:15:05.249000+0000 |                                                                     Masr7ya
 2026-04-02 16:15:17.300000+0000 |                                                                      Le feu
 2026-04-02 16:15:28.348000+0000 |                                                                      Krumla
 2026-04-02 16:15:35.390000+0000 |                                                                   Like I Do
 2026-04-02 16:15:47.440000+0000 |                                                               כאב של לוחמים
 2026-04-02 16:15:58.474000+0000 |                                                                      Sparks
 2026-04-02 16:16:13.515000+0000 |                                                           Never Broke Again
 2026-04-02 16:16:24.558000+0000 |                                                                      Casaya
 2026-04-02 16:16:36.604000+0000 |                                                          Y'a plus de raison
 2026-04-02 16:16:41.648000+0000 |                                                              PASEK PLAYBOYA
 2026-04-02 16:16:50.698000+0000 |                                                                       Dünya
 2026-04-02 16:17:05.748000+0000 |                                                                     Kuusamo
 2026-04-02 16:17:18.799000+0000 |                                                                   Šlamantys
 2026-04-02 16:17:28.845000+0000 |                                                                  drunk text
 2026-04-02 16:17:42.898000+0000 |                                            Hotel California - 2013 Remaster
 2026-04-02 16:17:56.944000+0000 |                                                                       Eagle
 2026-04-02 16:18:05.984000+0000 |                                                               Deck The Hall
 2026-04-02 16:18:12.029000+0000 |                                         Det är ju dej jag går och väntar på
 2026-04-02 16:18:22.070000+0000 |                                                 STOP & GO (feat. Billa Joe)
 2026-04-02 16:18:33.102000+0000 |                                                             אין לי מקום אחר
 2026-04-02 16:18:41.150000+0000 |                                                                    Pen Game
 2026-04-02 16:18:55.190000+0000 |                                                                   INTERLUDE
 2026-04-02 16:19:07.235000+0000 |                                                                La Campanera
 2026-04-02 16:19:15.282000+0000 |                                                         Dancing With Demons
 2026-04-02 16:19:24.327000+0000 |                                                              Die Bokmasjien
 2026-04-02 16:19:38.369000+0000 |                                                                   Unwritten
 2026-04-02 16:19:47.418000+0000 |                                                 Snow Flower (feat. Peakboy)
 2026-04-02 16:19:54.460000+0000 |                                                                     VICTORY
 2026-04-02 16:20:00.518000+0000 |                                                                       Theos
 2026-04-02 16:20:05.567000+0000 |                                                                   Nea Sezon
 2026-04-02 16:20:20.616000+0000 |                                                         GRISARNA TILL BACON
 2026-04-02 16:20:26.660000+0000 |                                                            Dýrð í dauðaþögn
 2026-04-02 16:20:33.704000+0000 |                                                                    Perfidia
 2026-04-02 16:20:43.743000+0000 |                                                                       Theos
 2026-04-02 16:20:50.778000+0000 |                              One Of The Girls (with JENNIE, Lily Rose Depp)
 2026-04-02 16:21:05.816000+0000 |                                                                    COLDBOYS
 2026-04-02 16:21:20.867000+0000 |                                                                      CHUPON
 2026-04-02 16:21:28.910000+0000 |                                                            Llegó la Navidad
 2026-04-02 16:21:39.962000+0000 |                                                              17 anni - skit
 2026-04-02 16:21:55.002000+0000 |                                                                     Sunrise
 2026-04-02 16:22:03.046000+0000 |                                                           Hie' Kommie Bokke
 2026-04-02 16:22:15.089000+0000 |                                                                       Oulfa
 2026-04-02 16:22:24.132000+0000 |                                                                   MOONLIGHT
 2026-04-02 16:22:29.171000+0000 |                                                                   Hora Loca
 2026-04-02 16:22:37.212000+0000 |                                                                    Ciclista
 2026-04-02 16:22:45.246000+0000 |                                                                  COSMETICOS
 2026-04-02 16:22:54.291000+0000 |                                          Hanggang Kailan - Umuwi Ka Na Baby
 2026-04-02 16:23:04.331000+0000 |                                                                       Ponya
 2026-04-02 16:23:16.370000+0000 |                                                                        Okej
 2026-04-02 16:23:23.411000+0000 |                                                                       Citi+
 2026-04-02 16:23:36.458000+0000 |                                                  Mindst Ondt (feat. Svea S)
 2026-04-02 16:23:45.497000+0000 |                                                                       Stroj
 2026-04-02 16:23:57.540000+0000 |                                                                         MMA
 2026-04-02 16:24:04.581000+0000 |                                                                      BeReal
 2026-04-02 16:24:12.625000+0000 |                                                          Aguinaldo Navideño
 2026-04-02 16:24:19.659000+0000 |                                                                      Hey Ho
 2026-04-02 16:24:24.690000+0000 |                                                                     Perillä
 2026-04-02 16:24:38.733000+0000 |                                                                    Joulumaa
 2026-04-02 16:24:45.768000+0000 |                        A.I.A. (Anna Inimesele Andeks) (feat. Lea Dali Lion)
 2026-04-02 16:24:51.805000+0000 |                                                                    Asfaltos
 2026-04-02 16:25:04.844000+0000 |                                                                   אהבה חולה
 2026-04-02 16:25:17.891000+0000 |                                                                      แอบหวัง
 2026-04-02 16:25:28.927000+0000 |                                                                    與我無關
 2026-04-02 16:25:40.972000+0000 |                                                                   Interaksi
 2026-04-02 16:25:56.024000+0000 |                                                 Diluvio (feat. Fight Pausa)
 2026-04-02 16:26:06.060000+0000 |                                                                 Empaquetate
 2026-04-02 16:26:16.098000+0000 |                                                                   Otherside
 2026-04-02 16:26:28.142000+0000 |                                                                       SUITE
 2026-04-02 16:26:38.182000+0000 |                                         Crazy - Baltimore Club Edition PT.1
 2026-04-02 16:26:49.225000+0000 |                                                            555 (feat. KESI)
 2026-04-02 16:26:58.266000+0000 |                                                                      Kamlee
 2026-04-02 16:27:09.303000+0000 |                                                                    Fast Car
 2026-04-02 16:27:21.346000+0000 |                                                                    Injabulo
 2026-04-02 16:27:32.380000+0000 |                                                                Dans le noir
 2026-04-02 16:27:45.423000+0000 |                                                                   Yakışıklı
 2026-04-02 16:27:56.457000+0000 |                                                                Zaļā Dziesma
 2026-04-02 16:28:06.507000+0000 |                                                                     Oh Shit
 2026-04-02 16:28:21.545000+0000 |                                                 Pakundangan (feat. Hev Abi)
 2026-04-02 16:29:26.267000+0000 |                                                             Pepernotensamba
 2026-04-02 16:29:40.328000+0000 |                                                       All for U (Ameyatchi)
 2026-04-02 16:29:54.365000+0000 |                                                         Spedizione punitiva
```